## 0. Imports

In [74]:
import os
os.environ["PYTHONWARNINGS"] = "ignore:pkg_resources is deprecated as an API:UserWarning"

import warnings
import random

from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import brier_score_loss
from sklearn.pipeline import Pipeline
import quantstats as qs
import pandas as pd
import numpy as np

from src.helpers.model_matrix import align_and_fill_dates_across_tickers
from src.helpers.model_matrix import build_model_matrix_from_wrds
from src.helpers._extract import ensure_dir

warnings.filterwarnings(
    "ignore",
    message=r"pkg_resources is deprecated as an API\..*",
    category=UserWarning,
)

warnings.filterwarnings(
    "ignore",
    message=r".*pkg_resources is deprecated as an API.*",
    category=UserWarning,
)

warnings.filterwarnings(
    "ignore",
    category=UserWarning,
    module=r"pkg_resources(\.|$)"
)

warnings.filterwarnings(
    "ignore",
    message="Mean of empty slice",
    category=RuntimeWarning,
)

## 1. Data build & cleaning (CRSP/DSF/IBES/FF)

In [75]:
tickers_list = [
    'AAPL', 'NVDA', 'MSFT', 'AMZN', 'TSLA',
     'GOOGL', 'LLY', 'WMT', 'JPM', 'BRK-B',
     #'V', 'MA', 'XOM', 'ORCL', 'UNH', 'COST', 'PG', 'HD', 'NFLX',
     #'JNJ', 'BAC', 'CRM', 'QQQ', 'ABBV', 'KO', 'CVX', 'TMUS', 'MRK', 'CSCO',
     #'WFC', 'ACN', 'NOW', 'TSM', 'AXP', 'PEP', 'MCD', 'IBM', 'MS', 'DIS',
     #'TMO', 'ABT', 'AMD', 'ADBE', 'PM', 'ISRG', 'GE', 'GS', 'INTU', 'CAT',
     #'TXN', 'QCOM', 'RY', 'VZ', 'DHR', 'BKNG', 'T', 'BLK', 'SPGI',
     #'RTX', 'PFE', 'NEE', 'HON', 'CMCSA', 'PGR', 'AMGN', 'LOW', 'ANET', 'UNP',
     #'SYK', 'TJX', 'C', 'BA', 'SCHW', 'BSX', 'KKR', 'ETN',
     #'COP', 'BX', 'PANW', 'ADP'
]

# will extract every possible ticker data (not just tickers_list) from wrds.
# the only wrds filters used are 'start' and 'end' dates.
# by extracting all possible ticker data, we can update tickers_list without
# connecting to wrds.
# the data are extracted from dsf, crsp, ff, and ibes (see src/migrations folder)
all_stocks = build_model_matrix_from_wrds(
    wrds_user="navid_namazi",
    start="2016-01-01",
    end="2021-01-01",
    chunk_size=500_000,
    tickers=tickers_list,
    use_run="last"  # "new", "last", or a specific folder name (i.e. "run_20250914_133747")
)

# ensure all stocks have the same date coverage
df = align_and_fill_dates_across_tickers(all_stocks=all_stocks)

[info] Using run folder: run_20251007_174249 (reuse=True)
[info] Reuse mode: all required Parquet files are present. No extraction performed.
{'run_folder': 'wrds_extracts\\run_20251007_174249', 'reuse': True, 'artifacts': {'dsf.parquet': 'wrds_extracts\\run_20251007_174249\\dsf.parquet', 'stocknames.parquet': 'wrds_extracts\\run_20251007_174249\\stocknames.parquet', 'ff.parquet': 'wrds_extracts\\run_20251007_174249\\ff.parquet', 'ibes_stats.parquet': 'wrds_extracts\\run_20251007_174249\\ibes_stats.parquet', 'ibes_act.parquet': 'wrds_extracts\\run_20251007_174249\\ibes_act.parquet'}}
[info] Removed 0 permnos(companies) for having zero in cfacpr or cfacshr
[info] Removed 0 permnos(companies) for exceeding the threshold of negative prices
$$$$ df_prices initial shape :  (11331, 18)
[info] ibes_stats: 274,942 (official_ticker, stat_date) pairs have >1 row (multiple horizons/periodicities). Will collapse before join.
[info] df_prices(+ibes): 95.2% missing in n_analysts.
[info] df_prices(+i

## 2. Train/test split & rolling CV split

In [76]:
from src.helpers.split_window import split_train_and_test
# Execute the split
random_state = 42
split_pct = 0.80
ins_dates, dates_out_sample, split_date = split_train_and_test(df, split_pct, random_state)

Total Data:
  Dates: 1189 trading days
  Period: 2016-04-14 to 2020-12-31
  Rows: 10,701 (stocks × dates)

Data Split:
   In-Sample (Development Set):
   Period: 2016-04-14 to 2020-01-23
   Dates: 951 days (80.0%)
   Purpose: feature selection, hyperparameter tuning, rolling CV

Out-Of-Sample (Test Set):
   Period: 2020-01-24 to 2020-12-31
   Dates: 238 days (20.0%)
   Purpose: final performance evaluation only

Split Date: 2020-01-24


In [77]:
from src.helpers.split_window import split_rolling_window

# Rolling window size configuration for in-sample (60/20/20 Split)
# When naming variables, ins short for in-sample, oos short for out-of-sample 

# Configure rolling windows
split_pct_rolling_train = 0.6  # 60% for training
split_pct_rolling_test = 0.2   # 20% for validation
target_folds_count = 10

ins_window_size, ins_training_window_size, ins_validation_window_size, step_size, actual_folds = split_rolling_window(
    ins_dates, 
    split_pct_rolling_train=split_pct_rolling_train,
    split_pct_rolling_test=split_pct_rolling_test,
    target_folds_count=target_folds_count
)

Rolling Window Configuration (In-Sample Only):
   Training window: 570 days (~2.3 years, 59.9% of in-sample)
   Validation window: 190 days (~0.8 years, 20.0% of in-sample)
   Step size: 21 days (~4.2 weeks)
   Target folds: 10
   Actual folds: 10
   Total validation observations: 1900 (across 10 overlapping folds)


## 3. Logistic Regression

### 3.1. Configuration

In [78]:
# Binary Target Definition
print("Target Column: adj_prc_logret_lead1 = next-day log return(t -> t+1)")
print("We predict: will the stock go up (1) or down (0) tomorrow?")

DIR_binary = (df["adj_prc_logret_lead1"] > 0).astype(int)  # 1 = up, 0 = down

# check class balance (market neutrality ~ 50/50)
print("\nBinary Target Distribution")
print(f" Up (1):   {(DIR_binary == 1).sum():,} observations ({(DIR_binary == 1).mean() * 100:.1f}%)")  # noqa
print(f" Down (0): {(DIR_binary == 0).sum():,} observations ({(DIR_binary == 0).mean() * 100:.1f}%)")  # noqa
print(f" Total:    {len(DIR_binary):,} observations")

# feature columns: everything except ticker, target, and the index columns, permno and date.
num_pred_cols = [c for c in df.columns if c not in (["ticker", "adj_prc_logret_lead1"])]
print(f"\nUsing {len(num_pred_cols)} features for prediction")

Target Column: adj_prc_logret_lead1 = next-day log return(t -> t+1)
We predict: will the stock go up (1) or down (0) tomorrow?

Binary Target Distribution
 Up (1):   5,743 observations (53.7%)
 Down (0): 4,958 observations (46.3%)
 Total:    10,701 observations

Using 102 features for prediction


### 3.2. Hyperparameter tuning: logistic regression + ElasticNet

In [79]:
print("ElasticNet Hyperparameters and Pipeline Configuration")

# ============================================================================
# HYPERPARAMETER TUNING CONFIGURATION
# ============================================================================
# Toggle between hardcoded hyperparameters (fast) and grid search tuning (slow)
TUNE_HYPERPARAMETERS = False  # Set to True to enable full grid search, False for hardcoded values

# Hardcoded hyperparameters (used when TUNE_HYPERPARAMETERS = False)
ELASTICNET_C = 0.1  # Inverse of regularization strength (higher C means less regularization, lower C means more regularization).
ELASTICNET_L1_RATIO = 0.7  # Controls the L1/L2 tradeoff (0 is ridge, 1 is lasso, anything in between mixes the two)

# Grid search ranges (used when TUNE_HYPERPARAMETERS = True)
# common grid search for l1_ratio : [0, 0.25, 0.5, 0.75, 1.0]  and for C: [0.01, 0.1, 1, 10, 100]
param_grid = {
    "clf__C": [0.01, 0.1, 1, 10],
    "clf__l1_ratio": [0.3, 0.5, 0.7],
}

print(f"\n🔧 Hyperparameter Tuning Mode: {'ENABLED (Grid Search)' if TUNE_HYPERPARAMETERS else 'DISABLED (Hardcoded)'}")
print(f"\nDefault/Hardcoded Hyperparameters:")
print(f"  C: {ELASTICNET_C}")
print(f"  l1_ratio: {ELASTICNET_L1_RATIO}")

if TUNE_HYPERPARAMETERS:
    print(f"\n📊 Grid Search will explore:")
    print(f"  C: {param_grid['clf__C']}")
    print(f"  l1_ratio: {param_grid['clf__l1_ratio']}")

# Preprocessing: Standardize features
ct = ColumnTransformer([(
    "num",  # numerical
    # scales each feature so ElasticNet penalty treats them similarly (to avoid leakage).
    # scaling is done per fold-by-fold
    StandardScaler(with_mean=True),
    num_pred_cols  # only numerical columns
)],
    remainder="drop",  # columns not listed are dropped
    sparse_threshold=0.0  # force dense for feature importance
)

# Classifier with ElasticNet regularization
clf = LogisticRegression(
    penalty="elasticnet",  # Use ElasticNet (L1 + L2)
    solver="saga",  # Only solver supporting elasticnet
    l1_ratio=ELASTICNET_L1_RATIO,  # Mix of L1 and L2
    C=ELASTICNET_C,  # Regularization strength
    max_iter=5000,  # More iterations for convergence
    tol=1e-4,
    random_state=random_state,
    n_jobs=-1  # Use all CPUs
)

# Pipeline: preprocessing to classification
pipe = Pipeline([("prep", ct), ("clf", clf)])

print("\nPipeline Flow:")
print("   1) 'prep' = ColumnTransformer(ct): standardize features (fit on train only to avoid leakage")
print("   2) 'clf'  = LogisticRegression with ElasticNet: learns β per fold")

ElasticNet Hyperparameters and Pipeline Configuration

🔧 Hyperparameter Tuning Mode: DISABLED (Hardcoded)

Default/Hardcoded Hyperparameters:
  C: 0.1
  l1_ratio: 0.7

Pipeline Flow:
   1) 'prep' = ColumnTransformer(ct): standardize features (fit on train only to avoid leakage
   2) 'clf'  = LogisticRegression with ElasticNet: learns β per fold


### 3.3. Rolling training & feature selection per fold (Model Selection and Cross-Validation)

In [80]:
print("Rolling Window Training With Feature Selection (In-Sample Only)")

pred_prob_up_new = pd.Series(index=df.index, dtype=float)
pred_prob_down_new = pd.Series(index=df.index, dtype=float)
pred_score_new = pd.Series(index=df.index, dtype=float)
pred_class_new = pd.Series(index=df.index, dtype=int)
used_mask_new = pd.Series(False, index=df.index)

# track feature selection across windows
feature_selection_history = []

window_num = 0
start_pos = 0

while start_pos + ins_training_window_size + ins_validation_window_size <= len(ins_dates):
    window_num += 1

    train_dates = ins_dates[start_pos: start_pos + ins_training_window_size]
    valid_dates = ins_dates[
                  start_pos + ins_training_window_size: start_pos + ins_training_window_size + ins_validation_window_size]

    print(f"\nWindow {window_num}: {train_dates.min().date()} to {valid_dates.max().date()}")

    start_pos += step_size

    idx_tr = df.index.get_level_values("date").isin(train_dates)
    idx_va = df.index.get_level_values("date").isin(valid_dates)

    X_tr = df.loc[idx_tr, num_pred_cols]
    y_tr = DIR_binary.loc[idx_tr]
    X_va = df.loc[idx_va, num_pred_cols]
    y_va = DIR_binary.loc[idx_va]
    groups_ins = X_tr.index.get_level_values("date")
    
    # Conditional hyperparameter tuning based on TUNE_HYPERPARAMETERS flag
    if TUNE_HYPERPARAMETERS:
        # Full grid search with cross-validation (slower)
        gs = GridSearchCV(
            estimator=pipe,
            param_grid=param_grid,
            scoring="neg_brier_score",  # maximize -Brier == minimize Brier
            cv=GroupKFold(n_splits=3),
            n_jobs=-1,
            refit=True  # refit on the whole training set with best params at the end of grid search and CV
        )
        gs.fit(X_tr, y_tr, groups=groups_ins)
        pipe_best = gs.best_estimator_
        best_C = gs.best_params_['clf__C']
        best_l1 = gs.best_params_['clf__l1_ratio']
        cv_score = gs.best_score_
    else:
        # Use hardcoded hyperparameters (faster)
        pipe_best = pipe
        pipe_best.fit(X_tr, y_tr)
        best_C = ELASTICNET_C
        best_l1 = ELASTICNET_L1_RATIO
        cv_score = None  # No CV score when not tuning

    # validation RMSE report (probability RMSE = sqrt(Brier))
    p_up_val = pipe_best.predict_proba(X_va)[:, 1]
    rmse_val = np.sqrt(brier_score_loss(y_va, p_up_val))
    print(f"   Best params: C={best_C}, l1={best_l1}")
    print(f"   Validation RMSE (proba): {rmse_val:.4f}")
    if cv_score is not None:
        print(f"   train(CV) RMSE (proba): {np.sqrt(np.sqrt(-cv_score)):.4f}")

    # feature Selection Analysis (from tuned model)
    coef = pipe_best.named_steps["clf"].coef_[0]

    feature_importance = pd.DataFrame({
        "feature": num_pred_cols,
        "coefficient": coef,
        "abs_coefficient": np.abs(coef)
    })

    ZERO_THRESHOLD = 1e-5
    selected_features = feature_importance[feature_importance["abs_coefficient"] > ZERO_THRESHOLD]
    n_selected = len(selected_features)
    pct_selected = (n_selected / len(num_pred_cols)) * 100

    top_features = selected_features.nlargest(10, "abs_coefficient")

    feature_selection_history.append({
        "window": window_num,
        "n_features_selected": n_selected,
        "pct_selected": pct_selected,
        "selected_features": selected_features["feature"].tolist(),
        "top_5_features": top_features.head(5)["feature"].tolist(),
        "train_start": train_dates.min(),
        "valid_end": valid_dates.max()
    })

    # predictions from the tuned model
    proba = pipe_best.predict_proba(X_va)
    p_down = proba[:, 0]
    p_up = proba[:, 1]
    score = 2 * p_up - 1
    yhat = (p_up > 0.5).astype(int)

    # store predictions
    pred_prob_up_new.loc[idx_va] = p_up
    pred_prob_down_new.loc[idx_va] = p_down
    pred_score_new.loc[idx_va] = score
    pred_class_new.loc[idx_va] = yhat
    used_mask_new.loc[idx_va] = True

    # report
    accuracy = (yhat == y_va).mean()  # noqa

    # High-confidence accuracy: only count predictions where p_up > 0.55 or p_up < 0.45
    high_conf_mask = (p_up > 0.55) | (p_up < 0.45)
    high_conf_accuracy = (yhat[high_conf_mask] == y_va.values[high_conf_mask]).mean()
    n_high_conf = high_conf_mask.sum()
    print(f"Training: {len(X_tr):,} samples")
    print(f"Validation: {len(X_va):,} samples, Accuracy: {accuracy:.1%}")
    print(f"High-confidence accuracy (p_up > 0.55) or (p_up < 0.45): {high_conf_accuracy:.1%} on {n_high_conf} samples")
    print("\nFeature Selection:")
    print(f"   Selected: {n_selected}/{len(num_pred_cols)} features ({pct_selected:.1f}%)")
    print("   Top 5 features by importance:")
    for i, row in top_features.head(5).iterrows():
        print(f"      {i + 1}. {row['feature']}: {row['coefficient']:+.4f}")

print(f"\nTraining complete: {window_num} windows processed")
print(f"Total validated: {used_mask_new.sum():,} / {len(df):,}")

# update global predictions with new ones
pred_prob_up = pred_prob_up_new
pred_prob_down = pred_prob_down_new
pred_score = pred_score_new
pred_class = pred_class_new
used_mask = used_mask_new

Rolling Window Training With Feature Selection (In-Sample Only)

Window 1: 2016-04-14 to 2019-04-22
   Best params: C=0.1, l1=0.7
   Validation RMSE (proba): 0.5412
Training: 5,130 samples
Validation: 1,710 samples, Accuracy: 45.6%
High-confidence accuracy (p_up > 0.55) or (p_up < 0.45): 44.1% on 1319 samples

Feature Selection:
   Selected: 75/102 features (73.5%)
   Top 5 features by importance:
      76. comm_ti_^VIX_atr_14: -0.2305
      7. comm_XLK_close_logret: -0.1854
      38. ratio_volatility_premium_lag1: -0.1838
      13. comm_^VXN_close_logret: -0.1652
      26. comm_^RUT_close_logret_lag5: +0.1627

Window 2: 2016-05-13 to 2019-05-21
   Best params: C=0.1, l1=0.7
   Validation RMSE (proba): 0.5307
Training: 5,130 samples
Validation: 1,710 samples, Accuracy: 46.9%
High-confidence accuracy (p_up > 0.55) or (p_up < 0.45): 46.6% on 1382 samples

Feature Selection:
   Selected: 73/102 features (71.6%)
   Top 5 features by importance:
      38. ratio_volatility_premium_lag1: -0.2

### 3.4. Feature selection across all folds

In [81]:
print("Feature Selection For Final Model")

# extract selection information from all windows
n_windows = len(feature_selection_history)
feature_counts = {feat: 0 for feat in num_pred_cols}

# count how many windows each feature was selected in
for window_info in feature_selection_history:
    selected_features = window_info["selected_features"]
    for feat in selected_features:
        feature_counts[feat] += 1

# calculate frequency (percentage of windows)
feature_freq = pd.DataFrame({
    "feature": list(feature_counts.keys()),
    "count": list(feature_counts.values()),
    "frequency": [count / n_windows for count in feature_counts.values()]
}).sort_values("frequency", ascending=False)

SELECTION_THRESHOLD = 0.5  # Features must appear in at least 50% of rolling windows

selected_features_mask = feature_freq["frequency"] >= SELECTION_THRESHOLD
final_feature_list = feature_freq.loc[selected_features_mask, "feature"].tolist()

print("\n  Selection Criterion:")
print(f"   Frequency threshold: ≥ {SELECTION_THRESHOLD * 100:.0f}% of windows")
print("   Rationale: Features must be consistently predictive across")
print("              different market regimes to be included")

print("\n  Results:")
print(f"   Features selected: {len(final_feature_list)} / {len(num_pred_cols)}")
print(f"   Features removed:  {len(num_pred_cols) - len(final_feature_list)}")
print(f"   Reduction: {(1 - len(final_feature_list) / len(num_pred_cols)) * 100:.1f}%")

# display selected features
print(f"\n  Selected Features ({len(final_feature_list)}):")
print(f"    {'Feature':<30} {'Frequency':<12}     {'Appearances':<15}")

selected_features_df = feature_freq[selected_features_mask].copy()
for idx, row in selected_features_df.iterrows():
    feat = row['feature']
    freq = row['frequency']
    count = row['count']
    bar = '█' * int(freq * 10)
    print(f"    {feat:<30} {freq:>6.1%} ({count:>2}/{n_windows})   {bar}")

# display removed features
removed_features_df = feature_freq[~selected_features_mask].copy()

if len(removed_features_df) > 0:
    print(f"\n  Removed Features ({len(removed_features_df)}):")
    print(f"    {'Feature':<30} {'Frequency':<12}     {'Appearances':<15}")

    for idx, row in removed_features_df.iterrows():
        feat = row['feature']
        freq = row['frequency']
        count = row['count']
        bar = '░' * int(freq * 10)
        print(f"    {feat:<30} {freq:>6.1%} ({count:>2}/{n_windows})   {bar}")


# feature categories breakdown
def categorize_feature(feat):
    """Categorize feature by type"""
    if feat.startswith("ti_"):
        return "Technical Indicator applied to ticker"
    if feat.startswith("comm_"):
        if "ti_" in feat:
            return "Technical Indicator applied to common feature"
        else:
            return "Common feature"
    if feat.startswith("ti_"):
        return "Technical Indicator applied to ticker"
    elif feat in ["mktrf", "smb", "hml", "rf", "umd"]:
        return "Fama-French Factors"
    elif feat.startswith("cons_") or feat.startswith("n_"):
        return "IBES Consensus"
    elif feat.startswith("adjclose_lag"):
        return "Price Lags"
    elif feat in ["adj_mktcap", "vol", "retx"]:
        return "Market Data"
    else:
        return "Other"


selected_features_df["category"] = selected_features_df["feature"].apply(categorize_feature)
category_counts = selected_features_df["category"].value_counts()

print("\n  Selected Features By Category:")

for category, count in category_counts.items():
    pct = count / len(selected_features_df) * 100
    print(f"    {category:<30} {count:>2} features ({pct:>5.1f}%)")

Feature Selection For Final Model

  Selection Criterion:
   Frequency threshold: ≥ 50% of windows
   Rationale: Features must be consistently predictive across
              different market regimes to be included

  Results:
   Features selected: 78 / 102
   Features removed:  24
   Reduction: 23.5%

  Selected Features (78):
    Feature                        Frequency        Appearances    
    adj_prc_logret                 100.0% (10/10)   ██████████
    adj_prc_logret_lag4            100.0% (10/10)   ██████████
    adj_prc_logret_lag1            100.0% (10/10)   ██████████
    adj_prc_logret_lag2            100.0% (10/10)   ██████████
    adj_prc_logret_lag5            100.0% (10/10)   ██████████
    comm_^VXN_close_logret         100.0% (10/10)   ██████████
    comm_XLF_close_logret_lag1     100.0% (10/10)   ██████████
    ratio_volatility_premium_lag1  100.0% (10/10)   ██████████
    comm_XLI_close_logret_lag1     100.0% (10/10)   ██████████
    comm_^RUT_close_logret_lag5    

### 3.5. Train the final in-sample model on all in-sample data

In [82]:
# hyperparameter selection on all in-sample

print("\nGlobal Hyperparameter Search (in-sample, using selected features)")

# build in-sample slice with final features
ins_mask = df.index.get_level_values("date").isin(ins_dates)
X_ins = df.loc[ins_mask, final_feature_list]
y_ins = DIR_binary.loc[ins_mask]

# group by date so folds split by day (avoid same-day leakage across stocks)
groups_ins = df.loc[ins_mask].index.get_level_values("date")


# rebuild pipeline constrained to the selected features
ct_final_cv = ColumnTransformer(
    [("num", StandardScaler(with_mean=True), final_feature_list)],
    remainder="drop",
    sparse_threshold=0.0
)
clf_final_cv = LogisticRegression(
    penalty="elasticnet",
    solver="saga",
    max_iter=5000,
    tol=1e-4,
    random_state=random_state,
    n_jobs=-1
)
pipe_final_cv = Pipeline([("prep", ct_final_cv), ("clf", clf_final_cv)])

# Conditional global hyperparameter tuning based on TUNE_HYPERPARAMETERS flag
if TUNE_HYPERPARAMETERS:
    # Full grid search with cross-validation (slower)
    param_grid_global = {
        "clf__C": [0.01, 0.1, 1, 10],
        "clf__l1_ratio": [0.3, 0.5, 0.7],
    }
    
    # time-aware CV via GroupKFold over dates
    cv_global = GroupKFold(n_splits=5)
    gs_global = GridSearchCV(
        estimator=pipe_final_cv,
        param_grid=param_grid_global,
        scoring="neg_brier_score",  # minimize Brier (probability RMSE)
        cv=cv_global,
        n_jobs=-1,
        refit=True
    )
    gs_global.fit(X_ins, y_ins, groups=groups_ins)
    
    best_C = gs_global.best_params_["clf__C"]
    best_l1 = gs_global.best_params_["clf__l1_ratio"]
    print(f"Best global params: C={best_C}, l1_ratio={best_l1}")
    print(f"Best mean CV Brier: {-gs_global.best_score_:.6f} (RMSE={(-gs_global.best_score_) ** 0.5:.4f})")
else:
    # Use hardcoded hyperparameters (faster)
    best_C = ELASTICNET_C
    best_l1 = ELASTICNET_L1_RATIO
    print(f"Using hardcoded params: C={best_C}, l1_ratio={best_l1}")

# lock in the global best hyperparameters and train final in-sample model
clf_final = LogisticRegression(
    penalty="elasticnet",
    solver="saga",
    C=best_C,
    l1_ratio=best_l1,
    max_iter=5000,
    tol=1e-4,
    random_state=random_state,
    n_jobs=-1
)
ct_final = ColumnTransformer(
    [("num", StandardScaler(with_mean=True), final_feature_list)],
    remainder="drop",
    sparse_threshold=0.0
)
pipe_final = Pipeline([("prep", ct_final), ("clf", clf_final)])
pipe_final.fit(X_ins, y_ins)

# report sparsity of the final model
final_coef = pipe_final.named_steps["clf"].coef_[0]
print(f"Final model non-zero coeffs: {(np.abs(final_coef) > 1e-5).sum()} / {len(final_coef)}")


Global Hyperparameter Search (in-sample, using selected features)
Using hardcoded params: C=0.1, l1_ratio=0.7
Final model non-zero coeffs: 65 / 78


## 4. Linear Regression

### 4.1. Configuration

In [83]:
# Continuous Target Definition for Linear Regression
print("Target Column: adj_prc_logret_lead1 = next-day log return(t -> t+1)")
print("We predict: the MAGNITUDE of tomorrow's return (continuous value)")

# Continuous target (actual log returns, not binarized)
y_continuous = df["adj_prc_logret_lead1"]

# Check distribution statistics
print("\nContinuous Target Distribution (Expected Returns)")
print(f"  Mean:        {y_continuous.mean():.6f}")
print(f"  Std Dev:     {y_continuous.std():.6f}")
print(f"  Min:         {y_continuous.min():.6f}")
print(f"  Max:         {y_continuous.max():.6f}")
print(f"  Median:      {y_continuous.median():.6f}")
print(f"  Total:       {len(y_continuous):,} observations")

# Distribution percentiles
print("\nPercentiles:")
percentiles = [1, 5, 10, 25, 50, 75, 90, 95, 99]
for p in percentiles:
    val = y_continuous.quantile(p / 100)
    print(f"  {p:>2}th:  {val:>10.6f}")

# Positive vs Negative returns (for context)
n_positive = (y_continuous > 0).sum()
n_negative = (y_continuous < 0).sum()
n_zero = (y_continuous == 0).sum()
print(f"\nDirection Split (for reference):")
print(f"  Positive returns: {n_positive:>5,} ({n_positive/len(y_continuous)*100:>5.1f}%)")
print(f"  Negative returns: {n_negative:>5,} ({n_negative/len(y_continuous)*100:>5.1f}%)")
print(f"  Zero returns:     {n_zero:>5,} ({n_zero/len(y_continuous)*100:>5.1f}%)")

# Confirm we're using the same feature set
print(f"\nUsing {len(num_pred_cols)} features for prediction (same as logistic regression)")


Target Column: adj_prc_logret_lead1 = next-day log return(t -> t+1)
We predict: the MAGNITUDE of tomorrow's return (continuous value)

Continuous Target Distribution (Expected Returns)
  Mean:        0.000654
  Std Dev:     0.045673
  Min:         -3.100480
  Max:         0.260876
  Median:      0.001098
  Total:       10,701 observations

Percentiles:
   1th:   -0.063688
   5th:   -0.030224
  10th:   -0.019431
  25th:   -0.006997
  50th:    0.001098
  75th:    0.009883
  90th:    0.020873
  95th:    0.031313
  99th:    0.066427

Direction Split (for reference):
  Positive returns: 5,743 ( 53.7%)
  Negative returns: 4,922 ( 46.0%)
  Zero returns:        36 (  0.3%)

Using 102 features for prediction (same as logistic regression)


### 4.2. Hyperparameter tuning: linear regression + ElasticNet


In [84]:
from sklearn.linear_model import ElasticNet
from sklearn.metrics import mean_squared_error, r2_score

print("=" * 80)
print("LINEAR REGRESSION MODEL: Expected Return Prediction")
print("=" * 80)

print("\n📈 Objective: Predict the magnitude of next-day returns (not just direction)")
print("   Target: adj_prc_logret_lead1 (continuous)")
print("   Purpose: Complement logistic regression for ensemble filtering\n")

# Linear regression will use ElasticNet (L1 + L2 regularization)
# Same regularization approach as logistic regression
# Use same hyperparameter tuning toggle as logistic regression

print("Linear Regression Hyperparameters and Pipeline Configuration")
print(f"\n🔧 Hyperparameter Tuning Mode: {'ENABLED (Grid Search)' if TUNE_HYPERPARAMETERS else 'DISABLED (Hardcoded)'}")

# Hardcoded hyperparameters for linear regression (ElasticNet)
# Note: For regression, alpha needs to be much smaller than for classification
# to avoid driving all coefficients to zero
LINEAR_ALPHA = 0.001  # ElasticNet regularization strength (higher = more regularization)
LINEAR_L1_RATIO = 0.5  # Mix of L1 and L2 (0=Ridge, 1=Lasso, 0.5=equal mix)

print(f"\nDefault/Hardcoded Hyperparameters:")
print(f"  alpha: {LINEAR_ALPHA}")
print(f"  l1_ratio: {LINEAR_L1_RATIO}")

# Grid search ranges (used when TUNE_HYPERPARAMETERS = True)
# Adjusted for regression problem - smaller alphas to avoid over-regularization
param_grid_linear = {
    "clf__alpha": [0.0001, 0.001, 0.01, 0.1],
    "clf__l1_ratio": [0.3, 0.5, 0.7, 0.9],
}

if TUNE_HYPERPARAMETERS:
    print(f"\n📊 Grid Search will explore:")
    print(f"  alpha: {param_grid_linear['clf__alpha']}")
    print(f"  l1_ratio: {param_grid_linear['clf__l1_ratio']}")

# Preprocessing: Standardize features (same as logistic)
ct_linear = ColumnTransformer([
    ("num", StandardScaler(with_mean=True), num_pred_cols)
], remainder="drop", sparse_threshold=0.0)

# ElasticNet regression with L1 + L2 regularization
clf_linear = ElasticNet(
    alpha=LINEAR_ALPHA,
    l1_ratio=LINEAR_L1_RATIO,
    max_iter=5000,
    tol=1e-4,
    random_state=random_state
)

# Pipeline: preprocessing to regression
pipe_linear = Pipeline([("prep", ct_linear), ("clf", clf_linear)])

print("\nPipeline Flow:")
print("   1) 'prep' = ColumnTransformer: standardize features")
print("   2) 'clf'  = ElasticNet Regression: predicts E[R] for t+1")


LINEAR REGRESSION MODEL: Expected Return Prediction

📈 Objective: Predict the magnitude of next-day returns (not just direction)
   Target: adj_prc_logret_lead1 (continuous)
   Purpose: Complement logistic regression for ensemble filtering

Linear Regression Hyperparameters and Pipeline Configuration

🔧 Hyperparameter Tuning Mode: DISABLED (Hardcoded)

Default/Hardcoded Hyperparameters:
  alpha: 0.001
  l1_ratio: 0.5

Pipeline Flow:
   1) 'prep' = ColumnTransformer: standardize features
   2) 'clf'  = ElasticNet Regression: predicts E[R] for t+1


### 4.3. Rolling training & feature selection per fold (Model Selection and Cross-Validation)

In [85]:
import time
from sklearn.base import clone

print("Rolling Window Training for Linear Regression (In-Sample Only)")
print("Using the SAME rolling windows as logistic regression")
print(f"Mode: {'GRID SEARCH' if TUNE_HYPERPARAMETERS else 'HARDCODED HYPERPARAMETERS (FAST)'}")
print(f"Hyperparameters: alpha={LINEAR_ALPHA}, l1_ratio={LINEAR_L1_RATIO}")
print("Note: If all features = 0, alpha is too high (decrease by 10x)\n")

pred_return_linear = pd.Series(index=df.index, dtype=float)
used_mask_linear = pd.Series(False, index=df.index)

# track feature selection across windows
feature_selection_history_linear = []

window_num = 0
start_pos = 0

while start_pos + ins_training_window_size + ins_validation_window_size <= len(ins_dates):
    window_num += 1
    window_start_time = time.time()

    train_dates = ins_dates[start_pos: start_pos + ins_training_window_size]
    valid_dates = ins_dates[
                  start_pos + ins_training_window_size: start_pos + ins_training_window_size + ins_validation_window_size]

    print(f"\n{'='*70}")
    print(f"Window {window_num}: {train_dates.min().date()} to {valid_dates.max().date()}")

    start_pos += step_size

    idx_tr = df.index.get_level_values("date").isin(train_dates)
    idx_va = df.index.get_level_values("date").isin(valid_dates)

    X_tr = df.loc[idx_tr, num_pred_cols]
    y_tr = y_continuous.loc[idx_tr]
    X_va = df.loc[idx_va, num_pred_cols]
    y_va = y_continuous.loc[idx_va]
    groups_ins = X_tr.index.get_level_values("date")
    
    print(f"Training samples: {len(X_tr):,} | Validation samples: {len(X_va):,}")
    
    # Conditional hyperparameter tuning based on TUNE_HYPERPARAMETERS flag
    fit_start_time = time.time()
    if TUNE_HYPERPARAMETERS:
        print("⚙️  Running GRID SEARCH with cross-validation...")
        # Full grid search with cross-validation (slower)
        gs_linear = GridSearchCV(
            estimator=pipe_linear,
            param_grid=param_grid_linear,
            scoring="neg_mean_squared_error",  # maximize -MSE == minimize MSE
            cv=GroupKFold(n_splits=3),
            n_jobs=-1,
            refit=True
        )
        gs_linear.fit(X_tr, y_tr, groups=groups_ins)
        pipe_best_linear = gs_linear.best_estimator_
        best_alpha = gs_linear.best_params_['clf__alpha']
        best_l1_ratio = gs_linear.best_params_['clf__l1_ratio']
        cv_score = gs_linear.best_score_
    else:
        print("⚡ Using HARDCODED hyperparameters (no grid search)...")
        # Use hardcoded hyperparameters (faster) - create fresh clone
        pipe_best_linear = clone(pipe_linear)
        pipe_best_linear.fit(X_tr, y_tr)
        best_alpha = LINEAR_ALPHA
        best_l1_ratio = LINEAR_L1_RATIO
        cv_score = None
    
    fit_time = time.time() - fit_start_time
    print(f"   Fitting time: {fit_time:.2f} seconds")

    # validation RMSE report
    y_pred_val = pipe_best_linear.predict(X_va)
    rmse_val = np.sqrt(mean_squared_error(y_va, y_pred_val))
    r2_val = r2_score(y_va, y_pred_val)
    
    print(f"   Best params: alpha={best_alpha}, l1_ratio={best_l1_ratio}")
    print(f"   Validation RMSE: {rmse_val:.6f}")
    print(f"   Validation R²: {r2_val:.4f}")
    if cv_score is not None:
        print(f"   train(CV) RMSE: {np.sqrt(-cv_score):.6f}")

    # feature Selection Analysis (from tuned model)
    coef = pipe_best_linear.named_steps["clf"].coef_

    feature_importance = pd.DataFrame({
        "feature": num_pred_cols,
        "coefficient": coef,
        "abs_coefficient": np.abs(coef)
    })

    ZERO_THRESHOLD = 1e-5
    selected_features = feature_importance[feature_importance["abs_coefficient"] > ZERO_THRESHOLD]
    n_selected = len(selected_features)
    pct_selected = (n_selected / len(num_pred_cols)) * 100

    top_features = selected_features.nlargest(10, "abs_coefficient")

    feature_selection_history_linear.append({
        "window": window_num,
        "n_features_selected": n_selected,
        "pct_selected": pct_selected,
        "selected_features": selected_features["feature"].tolist(),
        "top_5_features": top_features.head(5)["feature"].tolist(),
        "train_start": train_dates.min(),
        "valid_end": valid_dates.max()
    })

    # store predictions
    pred_return_linear.loc[idx_va] = y_pred_val
    used_mask_linear.loc[idx_va] = True

    # report
    print(f"\nFeature Selection:")
    print(f"   Selected: {n_selected}/{len(num_pred_cols)} features ({pct_selected:.1f}%)")
    print("   Top 5 features by importance:")
    for i, row in top_features.head(5).iterrows():
        print(f"      {i + 1}. {row['feature']}: {row['coefficient']:+.6f}")
    
    window_time = time.time() - window_start_time
    print(f"\n⏱️  Window {window_num} total time: {window_time:.2f} seconds ({window_time/60:.2f} minutes)")

print(f"\nLinear Regression Training complete: {window_num} windows processed")
print(f"Total validated: {used_mask_linear.sum():,} / {len(df):,}")


Rolling Window Training for Linear Regression (In-Sample Only)
Using the SAME rolling windows as logistic regression
Mode: HARDCODED HYPERPARAMETERS (FAST)
Hyperparameters: alpha=0.001, l1_ratio=0.5
Note: If all features = 0, alpha is too high (decrease by 10x)


Window 1: 2016-04-14 to 2019-04-22
Training samples: 5,130 | Validation samples: 1,710
⚡ Using HARDCODED hyperparameters (no grid search)...
   Fitting time: 0.11 seconds
   Best params: alpha=0.001, l1_ratio=0.5
   Validation RMSE: 0.023414
   Validation R²: -0.0012

Feature Selection:
   Selected: 14/102 features (13.7%)
   Top 5 features by importance:
      97. comm_ti_^GSPC_eom_14: -0.000390
      101. comm_ti_^GSPC_kurtosis_63: +0.000265
      21. comm_^GSPC_close_logret_lag1: -0.000242
      82. comm_ti_^VIX_psar_acc: -0.000237
      8. comm_XLF_close_logret: -0.000232

⏱️  Window 1 total time: 0.15 seconds (0.00 minutes)

Window 2: 2016-05-13 to 2019-05-21
Training samples: 5,130 | Validation samples: 1,710
⚡ Using HAR

### 4.4. Feature selection across all folds

In [86]:
print("Feature Selection For Linear Regression Model")

# extract selection information from all windows
n_windows_linear = len(feature_selection_history_linear)
feature_counts_linear = {feat: 0 for feat in num_pred_cols}

# count how many windows each feature was selected in
for window_info in feature_selection_history_linear:
    selected_features = window_info["selected_features"]
    for feat in selected_features:
        feature_counts_linear[feat] += 1

# calculate frequency (percentage of windows)
feature_freq_linear = pd.DataFrame({
    "feature": list(feature_counts_linear.keys()),
    "count": list(feature_counts_linear.values()),
    "frequency": [count / n_windows_linear for count in feature_counts_linear.values()]
}).sort_values("frequency", ascending=False)

SELECTION_THRESHOLD_LINEAR = 0.5  # Features must appear in at least 50% of rolling windows

selected_features_mask_linear = feature_freq_linear["frequency"] >= SELECTION_THRESHOLD_LINEAR
final_feature_list_linear = feature_freq_linear.loc[selected_features_mask_linear, "feature"].tolist()

print("\n  Selection Criterion:")
print(f"   Frequency threshold: ≥ {SELECTION_THRESHOLD_LINEAR * 100:.0f}% of windows")
print("   Rationale: Features must be consistently predictive across")
print("              different market regimes to be included")

print("\n  Results:")
print(f"   Features selected: {len(final_feature_list_linear)} / {len(num_pred_cols)}")
print(f"   Features removed:  {len(num_pred_cols) - len(final_feature_list_linear)}")
print(f"   Reduction: {(1 - len(final_feature_list_linear) / len(num_pred_cols)) * 100:.1f}%")

# Safety check: ensure at least some features were selected
if len(final_feature_list_linear) == 0:
    print("\n⚠️  WARNING: No features selected with current threshold!")
    print("   This usually means ElasticNet alpha is too high.")
    print("   Recommendations:")
    print("   1. Lower LINEAR_ALPHA (try 0.0001 or 0.00001)")
    print("   2. Lower l1_ratio (try 0.1 or 0.2 for more L2, less L1)")
    print("   3. Lower SELECTION_THRESHOLD_LINEAR temporarily")
    raise ValueError("No features selected for linear regression model. Adjust hyperparameters.")

# display selected features
print(f"\n  Selected Features ({len(final_feature_list_linear)}):")
print(f"    {'Feature':<30} {'Frequency':<12}     {'Appearances':<15}")

selected_features_df_linear = feature_freq_linear[selected_features_mask_linear].copy()
for idx, row in selected_features_df_linear.iterrows():
    feat = row['feature']
    freq = row['frequency']
    count = row['count']
    bar = '█' * int(freq * 10)
    print(f"    {feat:<30} {freq:>6.1%} ({count:>2}/{n_windows_linear})   {bar}")

# Compare with logistic regression features
print(f"\n  Comparison with Logistic Regression Features:")
logistic_features = set(final_feature_list)
linear_features = set(final_feature_list_linear)
common_features = logistic_features.intersection(linear_features)
logistic_only = logistic_features - linear_features
linear_only = linear_features - logistic_features

print(f"    Common to both models:    {len(common_features)} features")
print(f"    Logistic only:            {len(logistic_only)} features")
print(f"    Linear only:              {len(linear_only)} features")

if len(logistic_only) > 0 and len(logistic_only) <= 10:
    print(f"\n    Logistic-only features: {', '.join(list(logistic_only)[:10])}")
if len(linear_only) > 0 and len(linear_only) <= 10:
    print(f"    Linear-only features: {', '.join(list(linear_only)[:10])}")


Feature Selection For Linear Regression Model

  Selection Criterion:
   Frequency threshold: ≥ 50% of windows
   Rationale: Features must be consistently predictive across
              different market regimes to be included

  Results:
   Features selected: 13 / 102
   Features removed:  89
   Reduction: 87.3%

  Selected Features (13):
    Feature                        Frequency        Appearances    
    comm_XLF_close_logret          100.0% (10/10)   ██████████
    hml                            100.0% (10/10)   ██████████
    comm_ti_^GSPC_kurtosis_63      100.0% (10/10)   ██████████
    adj_prc_logret_lag4             90.0% ( 9/10)   █████████
    adj_prc_logret_lag2             80.0% ( 8/10)   ████████
    comm_ti_^GSPC_mfi_14            80.0% ( 8/10)   ████████
    comm_ti_^VIX_psar_acc           60.0% ( 6/10)   ██████
    ratio_XLV_^GSPC_lag1            60.0% ( 6/10)   ██████
    comm_ti_^GSPC_eom_14            60.0% ( 6/10)   ██████
    ratio_XLK_^GSPC                 50.0

### 4.5. Train the final in-sample model on all in-sample data

In [87]:
print("\nGlobal Hyperparameter Search for Linear Regression (in-sample, using selected features)")

# build in-sample slice with final features
ins_mask_linear = df.index.get_level_values("date").isin(ins_dates)
X_ins_linear = df.loc[ins_mask_linear, final_feature_list_linear]
y_ins_linear = y_continuous.loc[ins_mask_linear]

# group by date so folds split by day (avoid same-day leakage across stocks)
groups_ins_linear = df.loc[ins_mask_linear].index.get_level_values("date")

# rebuild pipeline constrained to the selected features
ct_final_linear = ColumnTransformer(
    [("num", StandardScaler(with_mean=True), final_feature_list_linear)],
    remainder="drop",
    sparse_threshold=0.0
)
clf_final_linear_cv = ElasticNet(
    max_iter=5000,
    tol=1e-4,
    random_state=random_state
)
pipe_final_linear_cv = Pipeline([("prep", ct_final_linear), ("clf", clf_final_linear_cv)])

# Conditional global hyperparameter tuning based on TUNE_HYPERPARAMETERS flag
if TUNE_HYPERPARAMETERS:
    # Full grid search with cross-validation (slower)
    param_grid_linear_global = {
        "clf__alpha": [0.0001, 0.001, 0.01, 0.1],
        "clf__l1_ratio": [0.3, 0.5, 0.7, 0.9],
    }
    
    # time-aware CV via GroupKFold over dates
    cv_global_linear = GroupKFold(n_splits=5)
    gs_global_linear = GridSearchCV(
        estimator=pipe_final_linear_cv,
        param_grid=param_grid_linear_global,
        scoring="neg_mean_squared_error",
        cv=cv_global_linear,
        n_jobs=-1,
        refit=True
    )
    gs_global_linear.fit(X_ins_linear, y_ins_linear, groups=groups_ins_linear)
    
    best_alpha_linear = gs_global_linear.best_params_["clf__alpha"]
    best_l1_ratio_linear = gs_global_linear.best_params_["clf__l1_ratio"]
    print(f"Best global params: alpha={best_alpha_linear}, l1_ratio={best_l1_ratio_linear}")
    print(f"Best mean CV MSE: {-gs_global_linear.best_score_:.8f} (RMSE={np.sqrt(-gs_global_linear.best_score_):.6f})")
else:
    # Use hardcoded hyperparameters (faster)
    best_alpha_linear = LINEAR_ALPHA
    best_l1_ratio_linear = LINEAR_L1_RATIO
    print(f"Using hardcoded params: alpha={best_alpha_linear}, l1_ratio={best_l1_ratio_linear}")

# lock in the global best hyperparameters and train final in-sample model
clf_final_linear = ElasticNet(
    alpha=best_alpha_linear,
    l1_ratio=best_l1_ratio_linear,
    max_iter=5000,
    tol=1e-4,
    random_state=random_state
)
ct_final_linear_final = ColumnTransformer(
    [("num", StandardScaler(with_mean=True), final_feature_list_linear)],
    remainder="drop",
    sparse_threshold=0.0
)
pipe_final_linear = Pipeline([("prep", ct_final_linear_final), ("clf", clf_final_linear)])
pipe_final_linear.fit(X_ins_linear, y_ins_linear)

# report model performance on in-sample
y_pred_ins = pipe_final_linear.predict(X_ins_linear)
rmse_ins = np.sqrt(mean_squared_error(y_ins_linear, y_pred_ins))
r2_ins = r2_score(y_ins_linear, y_pred_ins)

print(f"\nFinal Linear Model Performance (In-Sample):")
print(f"  RMSE: {rmse_ins:.6f}")
print(f"  R²: {r2_ins:.4f}")
print(f"  Non-zero coeffs: {(np.abs(pipe_final_linear.named_steps['clf'].coef_) > 1e-5).sum()} / {len(final_feature_list_linear)}")



Global Hyperparameter Search for Linear Regression (in-sample, using selected features)
Using hardcoded params: alpha=0.001, l1_ratio=0.5

Final Linear Model Performance (In-Sample):
  RMSE: 0.018090
  R²: 0.0049
  Non-zero coeffs: 5 / 13


## 5. Signal Confirmation & Trading Universe Selection

## 6. Allocation Strategy & Portfolio Construction

In [88]:
# ranking helper
def ranking_equal_weights(
    scores: pd.Series,
    long_threshold: float = 0.01,
    short_threshold: float = -0.01,
    max_long_pct: float = 0.20,
    max_short_pct: float = 0.20,
) -> pd.Series:
    """
    Threshold-based portfolio construction that respects ML model predictions.

    Args:
        scores: pd.Series indexed by stock identifier (e.g., permno) with model scores (e.g., in [-1, +1]).
        long_threshold: only consider stocks with score > long_threshold for longs.
        short_threshold: only consider stocks with score < short_threshold for shorts.
        max_long_pct: maximum fraction of the day's universe to allocate to longs.
        max_short_pct: maximum fraction of the day's universe to allocate to shorts.

    Returns:
        pd.Series of weights (same index as `scores`). Longs are positive , shorts are negative.
    """
    n = len(scores)
    w = pd.Series(0.0, index=scores.index)

    # Candidate selection by absolute threshold
    long_candidates = scores[scores > long_threshold].sort_values(ascending=False)
    short_candidates = scores[scores < short_threshold].sort_values(ascending=True)

    # Compute maximum allowed positions (allow zero)
    max_long_positions = int(max(1, np.floor(n * float(max_long_pct))))
    max_short_positions = int(max(1, np.floor(n * float(max_short_pct))))

    # Select up to the maximum allowed
    n_long_selected = min(len(long_candidates), max_long_positions)
    n_short_selected = min(len(short_candidates), max_short_positions)

    if n_long_selected > 0:
        sel_long = long_candidates.index[:n_long_selected]
        w.loc[sel_long] = 1.0 / n_long_selected

    if n_short_selected > 0:
        sel_short = short_candidates.index[:n_short_selected]
        w.loc[sel_short] = -1.0 / n_short_selected

    return w


# volatility helpers
def rolling_annualized_vol(series: pd.Series, window: int = 20) -> pd.Series:
    # std of daily log-returns, then annualize
    vol = series.rolling(window=window, min_periods=max(1, window // 2)).std()
    return vol * np.sqrt(252)


def attach_volatility(
        frame: pd.DataFrame,
        ret_col: str = "adj_prc_logret_lead1",
        idx_stock: str = "permno",
        idx_date: str = "date",
        window: int = 20,
        fill: str = "cross_section_median"
) -> pd.Series:
    # per-stock rolling vol
    vol = frame.groupby(level=idx_stock, group_keys=False)[ret_col].apply(
        lambda s: rolling_annualized_vol(s, window)
    )
    # cross-sectional fill
    if fill == "cross_section_median":
        vol = vol.groupby(level=idx_date).transform(lambda x: x.fillna(x.median()))
    return vol


# weighting transforms
def inverse_vol_adjust(raw_w: pd.Series, vol: pd.Series) -> pd.Series:
    return raw_w / vol


def normalize_dollar_neutral(
        w: pd.Series,
        long_target: float = 0.5,
        short_target: float = -0.5
) -> pd.Series:
    pos = w > 0
    neg = w < 0
    long_sum = w[pos].sum()
    short_sum = -w[neg].sum()
    out = w.copy()
    if long_sum > 0:
        out[pos] = w[pos] / long_sum * long_target
    if short_sum > 0:
        out[neg] = w[neg] / short_sum * (-short_target)
    return out


# performance helpers
def compute_simple_returns(logret: pd.Series) -> pd.Series:
    return np.exp(logret) - 1.0


def portfolio_returns(weights: pd.Series,
                      logret: pd.Series,
                      idx_date: str = "date") -> pd.Series:
    # weights are per (permno, date); multiply and sum by date
    daily = (weights * logret).groupby(level=idx_date).sum()
    return daily


def equity_from_simple_returns(simple_daily: pd.Series) -> pd.Series:
    return (1 + simple_daily).cumprod()


def stats_from_returns(simple_daily: pd.Series) -> dict:
    mu = simple_daily.mean()
    sd = simple_daily.std(ddof=0)
    sharpe = (mu / sd) * np.sqrt(252) if sd > 0 else np.nan
    eq = equity_from_simple_returns(simple_daily)
    total_ret = (eq.iloc[-1] - 1) * 100
    max_dd = (eq / eq.cummax() - 1).min() * 100
    return dict(
        trading_days=len(simple_daily),
        total_return=total_ret,
        sharpe=sharpe,
        max_drawdown=max_dd,
        avg_daily_ret=mu * 100,
        daily_vol=sd * 100,
        equity=eq
    )

In [89]:
def build_long_short_portfolio(
        frame: pd.DataFrame,
        score_col: str,
        ret_col: str = "adj_prc_logret_lead1",
        idx_date: str = "date",
        idx_stock: str = "permno",
        long_pct: float = 0.20,
        short_pct: float = 0.20,
        vol_window: int = 20,
        long_target: float = 0.5,
        short_target: float = -0.5,
        example_date_idx: int | None = 0,
        label: str = "Portfolio"
):
    # 1) ranking (equal-weight within buckets) per date
    w_raw = frame.groupby(level=idx_date, group_keys=False)[score_col] \
        .apply(lambda s: ranking_equal_weights(s, long_pct, short_pct))

    # 2) attach volatility and inverse-vol adjust
    vol = attach_volatility(frame, ret_col=ret_col, idx_stock=idx_stock, idx_date=idx_date, window=vol_window)
    w_vol = inverse_vol_adjust(w_raw, vol)

    # 3) normalize to dollar neutral per date
    w = w_vol.groupby(level=idx_date, group_keys=False).apply(
        lambda g: normalize_dollar_neutral(g, long_target=long_target, short_target=short_target)
    )

    # 4) diagnostics (optional prints)
    print(f"\nStrategy: {label} (top/bottom {int(long_pct * 100)}%)")
    print(f"  Total positions (non-zero): {(w != 0).sum():,}")  # noqa
    print(f"  Long positions:             {(w > 0).sum():,}")
    print(f"  Short positions:            {(w < 0).sum():,}")

    # one-day example
    if example_date_idx is not None:
        udates = frame.index.get_level_values(idx_date).unique()
        if 0 <= example_date_idx < len(udates):
            d = udates[example_date_idx]
            mask_d = frame.index.get_level_values(idx_date) == d
            w_d = w.loc[mask_d]
            print(f"\nExample for {pd.Timestamp(d).date()}:")
            print(f"  # longs:  {(w_d > 0).sum()} | sum longs:  {w_d[w_d > 0].sum():.4f}")
            print(f"  # shorts: {(w_d < 0).sum()} | sum shorts: {w_d[w_d < 0].sum():.4f}")
            print(f"  Net exposure: {w_d.sum():.4f}  (target ~ 0.00)")
            print(f"  Gross exposure: {w_d.abs().sum():.4f} (target ~ 1.00)")

    # 5) returns/metrics
    simple_daily = portfolio_returns(w, frame[ret_col], idx_date=idx_date)
    simple_daily = compute_simple_returns(simple_daily.apply(np.log1p))

    stats = stats_from_returns(simple_daily)
    print("\nPerformance:")
    print(f"  Trading days:  {stats['trading_days']}")
    print(f"  Total Return:  {stats['total_return']:7.2f}%")
    print(f"  Sharpe Ratio:  {stats['sharpe']:7.2f}")
    print(f"  Max Drawdown:  {stats['max_drawdown']:7.2f}%")
    print(f"  Avg Daily Ret: {stats['avg_daily_ret']:7.4f}%")
    print(f"  Daily Vol:     {stats['daily_vol']:7.4f}%")

    return dict(weights=w, vol=vol, daily_ret=simple_daily, equity=stats["equity"], stats=stats)

### 8. In-sample portfolio construction (Ranking-based trading strategy)

In [90]:
ins_df = df.loc[used_mask].copy()
ins_df["score"] = pred_score.loc[used_mask]

ins_result = build_long_short_portfolio(
    frame=ins_df,
    score_col="score",
    ret_col="adj_prc_logret_lead1",
    idx_date="date",
    idx_stock="permno",
    long_pct=0.20,
    short_pct=0.20,
    vol_window=20,
    long_target=0.5,
    short_target=-0.5,
    example_date_idx=10,
    label="In-sample (validation windows only)"
)


Strategy: In-sample (validation windows only) (top/bottom 20%)
  Total positions (non-zero): 571
  Long positions:             141
  Short positions:            349

Example for 2018-08-02:
  # longs:  1 | sum longs:  0.5000
  # shorts: 1 | sum shorts: -0.5000
  Net exposure: 0.0000  (target ~ 0.00)
  Gross exposure: 1.0000 (target ~ 1.00)

Performance:
  Trading days:  379
  Total Return:   -32.64%
  Sharpe Ratio:    -0.87
  Max Drawdown:   -40.34%
  Avg Daily Ret: -0.0905%
  Daily Vol:      1.6571%


## 7. Out-of-sample evaluation

In [91]:
print("Out-Of-Sample Evaluation")

# define oos slice
oos_mask = df.index.get_level_values("date").isin(dates_out_sample)

# X_test: the features selected by the stability+ElasticNet process
# y_test: the binary target (Up=1, Down=0)
X_test = df.loc[oos_mask, final_feature_list]
y_test = DIR_binary.loc[oos_mask]

print("\nOut-of-sample period:")
print(f"  Dates: {dates_out_sample.min().date()} to {dates_out_sample.max().date()}")
print(f"  Days: {len(dates_out_sample)}")
print(f"  Samples: {len(X_test):,}")

# score with the frozen final model
# - predict_proba: probability of Up tomorrow
# - score: signed score in [-1, +1] (centered probability)
# - predict: class label (0/1)
test_prob_up = pipe_final.predict_proba(X_test)[:, 1]
test_prob_down = 1 - test_prob_up
test_score = 2 * test_prob_up - 1
test_pred = pipe_final.predict(X_test)

# classification summary
test_accuracy = (test_pred == y_test).mean()  # noqa

print("\nClassification Performance:")
print(f"  Accuracy: {test_accuracy:.1%}")
print(f"  Up class accuracy: {(test_pred[y_test == 1] == 1).mean():.1%}")  # noqa
print(f"  Down class accuracy: {(test_pred[y_test == 0] == 0).mean():.1%}")  # noqa

# confusion matrix + detailed classification report
cm = confusion_matrix(y_test, test_pred, labels=[0, 1])
cm_df = pd.DataFrame(
    cm, index=["Actual Down (0)", "Actual Up (1)"],
    columns=["Pred Down (0)", "Pred Up (1)"]
)
print("\nConfusion Matrix:")
print(cm_df)

report = classification_report(
    y_test,
    test_pred,
    labels=[0, 1],
    target_names=["Down (0)", "Up (1)"],
    zero_division=0
)
print("\nClassification Report")
print(report)

pred_prob_up_test = pd.Series(test_prob_up, index=X_test.index, name="prob_up")
pred_score_test = pd.Series(test_score, index=X_test.index, name="score")
pred_class_test = pd.Series(test_pred, index=X_test.index, name="pred_class")

print(f"\nTest predictions created for {len(pred_score_test):,} observations")
print("\nBuilding OOS Long-Short Portfolio (Top/Bottom 20%)")

oos_df = df.loc[oos_mask].copy()
oos_df["score"] = pred_score_test
oos_df["pred_class"] = pred_class_test
oos_df["prob_up"] = pred_prob_up_test

oos_result = build_long_short_portfolio(
    frame=oos_df,
    score_col="score",
    ret_col="adj_prc_logret_lead1",
    idx_date="date",
    idx_stock="permno",
    long_pct=0.20,
    short_pct=0.20,
    vol_window=20,
    long_target=0.5,
    short_target=-0.5,
    example_date_idx=10,
    label="Out-of-sample"
)

stats = oos_result["stats"]
print("\nOOS Long-Short Headline Stats:")
print(f"  Total Return:   {stats.get('total_return', float('nan')):.2f}%")
print(f"  Sharpe Ratio:   {stats.get('sharpe', float('nan')):.2f}")
print(f"  Max Drawdown:   {stats.get('max_drawdown', float('nan')):.2f}%")
print(f"  Avg Daily Ret:  {stats.get('avg_daily_ret', float('nan')):.4f}%")
print(f"  Daily Vol:      {stats.get('daily_vol', float('nan')):.4f}%")

Out-Of-Sample Evaluation

Out-of-sample period:
  Dates: 2020-01-24 to 2020-12-31
  Days: 238
  Samples: 2,142

Classification Performance:
  Accuracy: 49.9%
  Up class accuracy: 48.1%
  Down class accuracy: 51.9%

Confusion Matrix:
                 Pred Down (0)  Pred Up (1)
Actual Down (0)            524          485
Actual Up (1)              588          545

Classification Report
              precision    recall  f1-score   support

    Down (0)       0.47      0.52      0.49      1009
      Up (1)       0.53      0.48      0.50      1133

    accuracy                           0.50      2142
   macro avg       0.50      0.50      0.50      2142
weighted avg       0.50      0.50      0.50      2142


Test predictions created for 2,142 observations

Building OOS Long-Short Portfolio (Top/Bottom 20%)

Strategy: Out-of-sample (top/bottom 20%)
  Total positions (non-zero): 360
  Long positions:             95
  Short positions:            184

Example for 2020-02-07:
  # longs:  0 | 

In [92]:
print("=" * 80)
print("ENSEMBLE EVALUATION: Combining Logistic + Linear Regression")
print("=" * 80)

# ============================================================================
# STEP 1: Linear Regression OOS Predictions
# ============================================================================
print("\n1️⃣  Linear Regression OOS Predictions")
print(f"   Making predictions for {len(X_test):,} samples...")

# Use the same X_test features, but with linear regression's selected features
X_test_linear = df.loc[oos_mask, final_feature_list_linear]

# Predict expected returns
test_expected_return = pipe_final_linear.predict(X_test_linear)

print(f"   Expected return predictions:")
print(f"     Mean: {test_expected_return.mean():.6f}")
print(f"     Std:  {test_expected_return.std():.6f}")
print(f"     Min:  {test_expected_return.min():.6f}")
print(f"     Max:  {test_expected_return.max():.6f}")

# Store predictions
pred_expected_return = pd.Series(test_expected_return, index=X_test.index, name="expected_return")

# ============================================================================
# STEP 2: Ensemble Agreement Analysis
# ============================================================================
print("\n2️⃣  Ensemble Agreement Analysis")

# Add predictions to oos_df
oos_df["expected_return"] = pred_expected_return

# Determine agreement
# Long signal: prob_up > 0.5 AND expected_return > 0
# Short signal: prob_up < 0.5 AND expected_return < 0
oos_df["logistic_signal"] = (oos_df["prob_up"] > 0.5).astype(int)  # 1 = bullish, 0 = bearish
oos_df["linear_signal"] = (oos_df["expected_return"] > 0).astype(int)  # 1 = bullish, 0 = bearish

oos_df["both_agree_up"] = (oos_df["logistic_signal"] == 1) & (oos_df["linear_signal"] == 1)
oos_df["both_agree_down"] = (oos_df["logistic_signal"] == 0) & (oos_df["linear_signal"] == 0)
oos_df["both_agree"] = oos_df["both_agree_up"] | oos_df["both_agree_down"]
oos_df["disagree"] = ~oos_df["both_agree"]

print(f"   Agreement Statistics:")
print(f"     Both agree UP:     {oos_df['both_agree_up'].sum():>5} ({oos_df['both_agree_up'].mean()*100:>5.1f}%)")
print(f"     Both agree DOWN:   {oos_df['both_agree_down'].sum():>5} ({oos_df['both_agree_down'].mean()*100:>5.1f}%)")
print(f"     Total agree:       {oos_df['both_agree'].sum():>5} ({oos_df['both_agree'].mean()*100:>5.1f}%)")
print(f"     Disagree:          {oos_df['disagree'].sum():>5} ({oos_df['disagree'].mean()*100:>5.1f}%)")

# ============================================================================
# STEP 3: Trading Strategy Configuration
# ============================================================================
print("\n3️⃣  Ensemble Trading Strategy Configuration")

# Option A: Rank by logistic score
# Option B: Rank by linear expected return
# Option C: Rank by combined score (weighted average)

RANKING_METHOD = "A"  # "A", "B", or "C"
WEIGHT_LOGISTIC = 0.7  # Used if RANKING_METHOD = "C"
WEIGHT_LINEAR = 0.3    # Used if RANKING_METHOD = "C"

print(f"   Ranking Method: {RANKING_METHOD}")
if RANKING_METHOD == "A":
    print(f"     Using logistic score only (prob_up)")
    oos_df["ensemble_score"] = oos_df["score"]
elif RANKING_METHOD == "B":
    print(f"     Using linear expected return only")
    # Convert expected return to [-1, +1] scale for consistency
    max_abs = oos_df["expected_return"].abs().max()
    oos_df["ensemble_score"] = oos_df["expected_return"] / max_abs if max_abs > 0 else 0.0
elif RANKING_METHOD == "C":
    print(f"     Using combined score (logistic: {WEIGHT_LOGISTIC}, linear: {WEIGHT_LINEAR})")
    # Normalize expected return to [-1, +1]
    max_abs = oos_df["expected_return"].abs().max()
    normalized_return = oos_df["expected_return"] / max_abs if max_abs > 0 else 0.0
    oos_df["ensemble_score"] = (WEIGHT_LOGISTIC * oos_df["score"] + 
                                 WEIGHT_LINEAR * normalized_return)

# ============================================================================
# STEP 4: Ensemble Filtering
# ============================================================================
print("\n4️⃣  Ensemble Filtering (Trade only when both models agree)")

# Set score to 0 for observations where models disagree
# This prevents trading on those days while keeping all dates in the output
oos_df.loc[oos_df["disagree"], "ensemble_score"] = 0.0

print(f"   Observations with non-zero scores: {(oos_df['ensemble_score'] != 0).sum():,}")
print(f"   Observations filtered out (disagreed): {(oos_df['ensemble_score'] == 0).sum():,}")

# ============================================================================
# STEP 5: Build Ensemble Portfolio
# ============================================================================
print("\n5️⃣  Building Ensemble Portfolio (Top/Bottom 20%)")

oos_result_ensemble = build_long_short_portfolio(
    frame=oos_df,
    score_col="ensemble_score",
    ret_col="adj_prc_logret_lead1",
    idx_date="date",
    idx_stock="permno",
    long_pct=0.20,
    short_pct=0.20,
    vol_window=20,
    long_target=0.5,
    short_target=-0.5,
    example_date_idx=10,
    label="Ensemble (Logistic + Linear)"
)

# ============================================================================
# STEP 6: Performance Comparison
# ============================================================================
print("\n6️⃣  Performance Comparison")
print("\n   Logistic Only vs Ensemble:")
print(f"   {'Metric':<20} {'Logistic Only':>15} {'Ensemble':>15} {'Difference':>15}")
print(f"   {'-'*20} {'-'*15} {'-'*15} {'-'*15}")

logistic_stats = oos_result["stats"]
ensemble_stats = oos_result_ensemble["stats"]

metrics = [
    ("Total Return %", "total_return"),
    ("Sharpe Ratio", "sharpe"),
    ("Max Drawdown %", "max_drawdown"),
    ("Avg Daily Ret %", "avg_daily_ret"),
    ("Daily Vol %", "daily_vol"),
]

for label, key in metrics:
    log_val = logistic_stats.get(key, float('nan'))
    ens_val = ensemble_stats.get(key, float('nan'))
    diff = ens_val - log_val
    print(f"   {label:<20} {log_val:>15.4f} {ens_val:>15.4f} {diff:>15.4f}")

print("\n✅ Ensemble evaluation complete!")
print(f"   oos_result_ensemble created successfully")


ENSEMBLE EVALUATION: Combining Logistic + Linear Regression

1️⃣  Linear Regression OOS Predictions
   Making predictions for 2,142 samples...
   Expected return predictions:
     Mean: 0.000984
     Std:  0.001667
     Min:  -0.004189
     Max:  0.012693

2️⃣  Ensemble Agreement Analysis
   Agreement Statistics:
     Both agree UP:       887 ( 41.4%)
     Both agree DOWN:     290 ( 13.5%)
     Total agree:        1177 ( 54.9%)
     Disagree:            965 ( 45.1%)

3️⃣  Ensemble Trading Strategy Configuration
   Ranking Method: A
     Using logistic score only (prob_up)

4️⃣  Ensemble Filtering (Trade only when both models agree)
   Observations with non-zero scores: 1,177
   Observations filtered out (disagreed): 965

5️⃣  Building Ensemble Portfolio (Top/Bottom 20%)

Strategy: Ensemble (Logistic + Linear) (top/bottom 20%)
  Total positions (non-zero): 362
  Long positions:             83
  Short positions:            198

Example for 2020-02-07:
  # longs:  0 | sum longs:  0.0000
 

## 8. Output Generation

In [93]:
from src.helpers.output_generation import generate_oos_report

# Generate HTML report for the ensemble strategy
generate_oos_report(
    portfolio_result=oos_result_ensemble,
    oos_df=oos_df,
    dates_out_sample=dates_out_sample,
    output_path="out/oos_ensemble_long_short_tearsheet.html",
    report_title=f"OUT-OF-SAMPLE ENSEMBLE: Long-Short Market Neutral ({dates_out_sample.min().date()} to {dates_out_sample.max().date()})",
    output_dir="out"
)

Generating Out-Of-Sample HTML Report
    Saved: out/oos_ensemble_long_short_tearsheet.html
   Period: 2020-01-27 to 2020-12-31
   Days:   237
